# SQL for Data Platforms
## Part 2: Designing the Schema — Star vs. Snowflake

Part 1 built a normalized, transactional (OLTP) schema — `customers`, `products`, `orders`,
`order_items`, `employees` — good for fast, consistent writes. This part asks a different
question: if you were building this as an **analytical** schema instead (a warehouse, not an
app database), what would the tables look like?

For the architectural *why* (fact tables, dimensions, when this pattern beats a single flat
table), see [Dimensional Modeling & the Star Schema](../../mfzamudio.github.io/publications/pattern-dimensional-modeling.html)
on the main site — this notebook is the *how do I write the DDL* companion to that page.

## Setup

A fresh in-memory database for this module — we're modeling a **new**, analytical schema here,
not reusing Part 1's OLTP tables directly (that's the point: the shapes are different).

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')

def sql(query):
    return pd.read_sql_query(query, conn)

def execute(statement):
    conn.execute(statement)
    conn.commit()

print("fresh in-memory database ready")

fresh in-memory database ready


## Section 1 — The Star Schema

A **fact table** holds *what happened* — one row per event/transaction, mostly foreign keys and
measures (numbers). **Dimension tables** hold *the descriptive angles* you slice by — fewer rows,
richer text columns. Put one fact table in the middle with dimensions around it, and it looks like
a star.

**Surrogate keys:** dimension tables get their own auto-incrementing integer key
(`customer_key`, not the source system's `customer_id`) — this decouples the warehouse from the
source system's ID and, critically, is what makes Part 8's Slowly Changing Dimensions possible
(a customer can have multiple rows over time, one surrogate key each, all sharing the same
source `customer_id`).

In [2]:
# DIMENSION: dim_customer — one row per customer (denormalized: city lives here, not in a
# separate "cities" table — that would be the SNOWFLAKE version, see Section 2)
execute("""
CREATE TABLE dim_customer (
    customer_key     INTEGER PRIMARY KEY,   -- surrogate key
    customer_id      INTEGER NOT NULL,      -- source system's natural key
    name             VARCHAR(100) NOT NULL,
    city             VARCHAR(50),
    membership_level VARCHAR(10)
)
""")

# DIMENSION: dim_product
execute("""
CREATE TABLE dim_product (
    product_key INTEGER PRIMARY KEY,
    product_id  INTEGER NOT NULL,
    name        VARCHAR(100) NOT NULL,
    category    VARCHAR(50) NOT NULL
)
""")

# DIMENSION: dim_date — a pre-built date spine, standard in every star schema
execute("""
CREATE TABLE dim_date (
    date_key   INTEGER PRIMARY KEY,   -- YYYYMMDD as an integer, e.g. 20230115
    full_date  DATE NOT NULL,
    year       INTEGER NOT NULL,
    month      INTEGER NOT NULL,
    month_name VARCHAR(10) NOT NULL,
    quarter    INTEGER NOT NULL
)
""")

# FACT: fact_sales — one row per order line item; mostly keys + measures
execute("""
CREATE TABLE fact_sales (
    sales_key    INTEGER PRIMARY KEY,
    customer_key INTEGER NOT NULL REFERENCES dim_customer(customer_key),
    product_key  INTEGER NOT NULL REFERENCES dim_product(product_key),
    date_key     INTEGER NOT NULL REFERENCES dim_date(date_key),
    quantity     INTEGER NOT NULL,
    unit_price   DECIMAL(10,2) NOT NULL,
    revenue      DECIMAL(10,2) NOT NULL
)
""")
print("star schema created: dim_customer, dim_product, dim_date, fact_sales")

star schema created: dim_customer, dim_product, dim_date, fact_sales


In [3]:
execute("INSERT INTO dim_customer VALUES (1, 1, 'Alice Martin', 'Toronto', 'premium')")
execute("INSERT INTO dim_customer VALUES (2, 7, 'Grace Kim', 'Vancouver', 'vip')")
execute("INSERT INTO dim_product VALUES (1, 1, 'Wireless Headphones', 'Electronics')")
execute("INSERT INTO dim_product VALUES (2, 13, 'Python Programming', 'Books')")
execute("INSERT INTO dim_date VALUES (20230120, '2023-01-20', 2023, 1, 'January', 1)")
execute("""
INSERT INTO fact_sales VALUES
    (1, 1, 1, 20230120, 1, 129.99, 129.99),
    (2, 2, 2, 20230120, 2,  44.99,  89.98)
""")

# A star-schema query: revenue by city and category — join the fact to every dimension it needs
sql("""
SELECT c.city, p.category, d.month_name, SUM(f.revenue) AS revenue
FROM fact_sales f
JOIN dim_customer c ON c.customer_key = f.customer_key
JOIN dim_product  p ON p.product_key  = f.product_key
JOIN dim_date     d ON d.date_key     = f.date_key
GROUP BY c.city, p.category, d.month_name
""")

,city,category,month_name,revenue
0,Toronto,Electronics,January,129.99
1,Vancouver,Books,January,89.98


## Section 2 — Snowflake: normalizing a dimension further

`dim_customer` above is denormalized — `city` sits directly on the customer row. **Snowflaking**
means splitting a dimension out into its own sub-table when its attributes have their own
hierarchy worth modeling separately (e.g. city → province → country), at the cost of an extra
JOIN on every query that needs it.

**When to snowflake:** the sub-attribute is reused by many dimension rows and changes
independently (a city's province doesn't change, but 500 customers might share one city) — this
saves storage and keeps that hierarchy consistent in one place. **When to keep it a star:** the
attribute is cheap to duplicate and rarely queried on its own — most modern column-oriented
warehouses compress repeated text so well that the storage savings from snowflaking rarely
justify the extra JOIN. **Star is the default; snowflake only when there's a real reason.**

In [4]:
execute("""
CREATE TABLE dim_city (
    city_key INTEGER PRIMARY KEY,
    city     VARCHAR(50) NOT NULL,
    province VARCHAR(50) NOT NULL,
    country  VARCHAR(50) NOT NULL DEFAULT 'Canada'
)
""")
execute("""
CREATE TABLE dim_customer_snowflaked (
    customer_key INTEGER PRIMARY KEY,
    customer_id  INTEGER NOT NULL,
    name         VARCHAR(100) NOT NULL,
    city_key     INTEGER REFERENCES dim_city(city_key),
    membership_level VARCHAR(10)
)
""")
execute("INSERT INTO dim_city VALUES (1, 'Toronto', 'Ontario', 'Canada')")
execute("INSERT INTO dim_customer_snowflaked VALUES (1, 1, 'Alice Martin', 1, 'premium')")

# Same question, one more JOIN required
sql("""
SELECT cust.name, city.city, city.province
FROM dim_customer_snowflaked cust
JOIN dim_city city ON city.city_key = cust.city_key
""")

,name,city,province
0,Alice Martin,Toronto,Ontario


## Best Practices — Schema Design

- Default to a star schema. Snowflake a dimension only when you have a concrete reason (a
  genuinely reused, independently-changing hierarchy) — not "because it's more normalized."
- Always give dimensions a surrogate key, separate from the source system's natural key. This is
  a prerequisite for Slowly Changing Dimensions (Part 8) — you cannot version a row's history if
  its key *is* the natural key.
- Build a `dim_date` table once and reuse it everywhere — computing month/quarter/fiscal-period
  logic in every query is both slower and a source of subtle inconsistency.

## Next

**Part 3 — Joins & Aggregations** returns to querying, using the original Part 1 schema (this
part's star schema comes back explicitly in **Part 8 — Slowly Changing Dimensions**, where
`dim_customer` gains real version history).